# Resource interchange with QREF and Bartiq

Export a native alias-preparation model to QREF v1, then use Bartiq to evaluate and replace a child cost. The lesson uses QREF 0.11.0, Bartiq 0.12.1, and the Graphviz executable.

## Basics

### Prepare an alias-sampling table

In [ ]:
from IPython.display import SVG, display

from examples.algorithms.resource_interchange_catalog import alias_interchange_example
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

alias = alias_interchange_example()
graph = alias.call_graph()
dot = export_call_graph_dot(
    graph.root,
    graph.direct_calls,
    node_properties=lambda name: {
        "label": name, "shape": "box", "style": "rounded,filled",
        "fillcolor": "#eef5f8", "fontname": "Helvetica", "margin": "0.18,0.10",
    },
)
display(SVG(data=render_call_graph_svg(dot)))

### Export the typed resource graph to QREF

In [ ]:
from qref.experimental.rendering import to_graphviz
from qref.verification import verify_topology

from examples.algorithms.resource_interchange_catalog import alias_preparation_qref

qref_definition = alias_preparation_qref(alias)
assert verify_topology(qref_definition)
display(SVG(data=to_graphviz(qref_definition).pipe(format="svg")))

### Compare the model and exported resources

In [ ]:
from flagquantum.algorithms.primitives.alias_preparation import source_dense_alias_preparation_cost
from flagquantum.compiler._component_trace.gate_budget import legacy_gate_budget
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources

shims = source_dense_alias_preparation_cost(alias.table, legacy_shims=True)
historical = legacy_gate_budget(estimate_component_resources(shims).gates)
exported = {resource.name: resource.value for resource in qref_definition.program.resources}
assert historical == exported == {"t": 307, "clifford": 1770, "rotations": 2}
default_budget = legacy_gate_budget(estimate_component_resources(alias.resource_model()).gates)
print("Pinned historical budget", historical)
print("Native default budget", default_budget)

Each child has its own budget. Inspect uniform preparation before summing the complete graph.

In [ ]:
uniform_child = qref_definition.program.children.by_name["PrepareUniformSuperposition_0"]
{resource.name: resource.value for resource in uniform_child.resources}

### Compile the child costs with Bartiq

In [ ]:
from bartiq import compile_routine

decomposed_model = qref_definition.model_copy(deep=True)
decomposed_model.program.resources = []
compiled_alias = compile_routine(decomposed_model).routine
assert {name: resource.value for name, resource in compiled_alias.resources.items()} == historical
compiled_alias.resources

## A custom uniform preparation cost

### Use a symbolic alias shape

In [ ]:
from sympy import symbols

from flagquantum.algorithms.primitives.symbolic_alias_preparation import AliasPreparationShape

N, M, eps = symbols("N M eps")
symbolic_alias = AliasPreparationShape.from_precision(N, M, eps)
symbolic_alias.resource_model(legacy_shims=True)

The table-free shape retains the probability precision, normalization and register widths as expressions.

### Export the pinned symbolic uniform model

In [ ]:
from examples.algorithms.resource_interchange_catalog import uniform_preparation_qref

pinned_uniform = uniform_preparation_qref(N)
pinned_uniform.program.resources

The pinned T expression is `12·ceil(log2(floor(N))) − 4`. The uniform-preparation analysis uses the odd part of `N` and gives a different estimate. Both formulas here describe models.

### Declare the alternative cost in QREF

In [ ]:
from qref import SchemaV1

port_size = (alias.table.selection_size - 1).bit_length()
custom_uniform = SchemaV1(
    version="v1",
    program={
        "name": "PrepareUniformSuperposition",
        "input_params": ["N"],
        "ports": [
            {"name": "in_target", "direction": "input", "size": port_size},
            {"name": "out_target", "direction": "output", "size": port_size},
        ],
        "resources": [
            {"name": "t", "type": "additive", "value": "10*ceiling(log2(L)) + 2*k"},
            {"name": "rotations", "type": "additive", "value": "2"},
        ],
        "local_variables": {"k": "multiplicity(2, N)", "L": "N/2**k"},
    },
)

The alternative specifies T gates and rotations. Its Clifford cost is unspecified.

In [ ]:
for name, model in (("Pinned uniform model", pinned_uniform), ("Alternative uniform model", custom_uniform)):
    print(name, compile_routine(model).routine.resources["t"].value)

## Replace uniform preparation in the alias model

In [ ]:
updated_alias = alias_preparation_qref(alias)
updated_alias.program.resources = []
[child.name for child in updated_alias.program.children]

The child identifier includes its occurrence index. Preserve that identifier when replacing the routine so its parent connections still resolve.

In [ ]:
child_name = "PrepareUniformSuperposition_0"
replacement = custom_uniform.program.model_copy(deep=True)
replacement.name = child_name
updated_alias.program.children.by_name[child_name] = replacement
compiled_custom = compile_routine(updated_alias).routine
print("Pinned alias T count", compiled_alias.resources["t"].value)
print("Updated alias T expression", compiled_custom.resources["t"].value)

The original notebook used the root assignment name `N`. The compiled cost contains the nested parameter `PrepareUniformSuperposition_0.N`, which requires a qualified assignment for numeric evaluation.

In [ ]:
from bartiq import evaluate

population = alias.table.selection_size
root_assignment = evaluate(compiled_custom, {"N": population}).routine.resources["t"].value
qualified_assignment = evaluate(
    compiled_custom, {f"{child_name}.N": population}
).routine.resources["t"].value
assert qualified_assignment == 313
print("Root assignment", root_assignment)
print("Qualified assignment", qualified_assignment)

## Custom cost functions

In [ ]:
def foo(n):
    return n if n < 10 else foo(n - 3)

In [ ]:
functional_uniform = custom_uniform.model_copy(deep=True)
functional_uniform.program.resources = [
    {"name": "t", "type": "additive", "value": "2*foo(N)"}
]
compiled_function = compile_routine(functional_uniform).routine
compiled_function.resources

Bartiq passes the symbolic function argument to a callable supplied by the notebook.

In [ ]:
from collections import Counter

from flagquantum.circuit import Circuit

custom = evaluate(compiled_function, {"N": 101}, functions_map={"foo": foo})
assert custom.routine.resources["t"].value == 16
circuit = Circuit(alias.qubit_count).append(alias)
print("Custom uniform T model at N=101", custom.routine.resources["t"].value)
print("Emitted alias primitives", dict(Counter(gate.name for gate in circuit.to_ir().instructions)))

The function evaluation changes the symbolic budget. Primitive instruction counts come from the separately appended native circuit.

## Closing remarks

QREF captures ports, connections and declared resources. Bartiq evaluates those declarations independently of the circuit execution path.